# 01 トークナイザーを観察する

言語モデルはテキストをそのまま処理するのではなく、**トークン**という単位に分割してから処理します。

このノートブックでは、トークナイザーの動作を手を動かしながら確認します。LLM動かすだけなら気にしなくてもよいようなバイト列の詳細まで見てみます。

## このノートブックでやること

0. **前提：文字コードのしくみ** — 16 進数・Unicode・UTF-8 を実例で確認する
1. **トークナイザーの読み込み** — Qwen3-4B のトークナイザーを準備する
2. **encode / decode とトークン分割** — 基本操作を確認し、様々なテキストの分割を観察する
3. **special token** — `<|im_start|>` などのシステム用トークンを確認する
4. **chat template の適用とトークン化** — メッセージがどんなトークン列になるかを表で見る
5. **語彙の先頭と末尾** — token ID の順序と語彙の構造を覗く

## トークナイザーの役割

```
テキスト
  ↓ tokenizer.encode()   # テキスト → token ID 列
token ID 列（整数の配列）
  ↓ model(...)           # Transformer が処理
token ID 列（生成結果）
  ↓ tokenizer.decode()   # token ID 列 → テキスト
テキスト（応答）
```

Qwen3-4B の `tokenizer.vocab_size` は **151,643** です。  
これに `<|im_start|>` などの（special token を含む）追加トークン 26 個が加わり、`len(tokenizer)` = **151,669** になります。  
（モデルの embedding 層は GPU 効率のため 151,936 スロット持つが、そのうち 267 スロットは未使用）

## 0. 環境セットアップ（3環境 自動切替: Colabだけ pip / path も自動）


In [1]:
# 3環境(Mac/Win/Colab)を同一ファイルで動かすための判定。Colabのみ pip（Mac/Winはenvに在るのでskip）。
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "transformers==5.9.0" "accelerate==1.13.0"
    print("Colab: pip done")
else:
    print("local(Mac/Win): pip skip（env利用）")

local(Mac/Win): pip skip（env利用）


---
## 0. 前提：文字コードのしくみ

テキストはコンピューター内部では **数値の列** として扱われます。  
このノートブックに登場する `hex_bytes`・`piece`・`U+XXXX` の意味を理解するために、  
16 進数 → Unicode → UTF-8 の 3 つの概念を実例で確認します。

### 16 進数（hex）

In [2]:
# 10 進数と 16 進数は同じ値の別表記
# 1 バイト = 8 ビット = 0〜255 の整数 → 2 桁の 16 進数で表すのが慣習
print(0x41)        # 16 進数リテラル → 10 進数で表示
print(hex(65))     # 10 進数 → 16 進数文字列
print(0xFF)        # 1 バイトの最大値

# 確認：0x41 と 65 と 'A' はすべて同じものを指す
print(0x41 == 65)
print(chr(0x41))   # 整数をその Unicode 文字に変換

65
0x41
255
True
A


### Unicode とコードポイント

**Unicode** は「世界中のすべての文字に番号を割り当てる規則」です。  
その番号を **コードポイント** と呼び、`U+XXXX`（4 桁以上の 16 進数）と書きます。  
`U+` は慣習的な接頭辞で、`U+0041` = `0x0041` = `65` はすべて同じ値です。

In [3]:
# ord(): 文字 → コードポイント（整数）
# chr(): コードポイント（整数） → 文字

for ch in ['!', 'A', 'é', 'δ', 'あ', '京', '🤖']:
    cp = ord(ch)
    print(f"  ord({ch!r}) = {cp:#06x} = {cp:6d},  chr({cp}) = {chr(cp)!r}")

  ord('!') = 0x0021 =     33,  chr(33) = '!'
  ord('A') = 0x0041 =     65,  chr(65) = 'A'
  ord('é') = 0x00e9 =    233,  chr(233) = 'é'
  ord('δ') = 0x03b4 =    948,  chr(948) = 'δ'
  ord('あ') = 0x3042 =  12354,  chr(12354) = 'あ'
  ord('京') = 0x4eac =  20140,  chr(20140) = '京'
  ord('🤖') = 0x1f916 = 129302,  chr(129302) = '🤖'


### UTF-8 エンコーディング

**エンコーディング**とは、コードポイント（番号）を実際の **バイト列** に変換する方式です。  
最も広く使われる **UTF-8** は可変長で、文字の種類によって使うバイト数が変わります。

| コードポイントの範囲 | バイト数 | 代表例 |
|---|:---:|---|
| U+0000 〜 U+007F | 1 | ASCII：英数字・記号（`A` `!` `0`） |
| U+0080 〜 U+07FF | 2 | ラテン拡張・ギリシャ文字など（`é` `ñ` `α`） |
| U+0800 〜 U+FFFF | 3 | 日本語・中国語・韓国語など（`あ` `京` `한`） |
| U+10000 〜 U+10FFFF | 4 | 絵文字・稀少文字など（`🤖` `🎌`） |

#### 変換規則（ビットパターン）

バイト数によって、先頭バイトの書き方が決まっています。  
`x` の部分にコードポイントのビットを詰めます。

```
1バイト: 0xxxxxxx
2バイト: 110xxxxx  10xxxxxx
3バイト: 1110xxxx  10xxxxxx  10xxxxxx
4バイト: 11110xxx  10xxxxxx  10xxxxxx  10xxxxxx
```

続きバイトは必ず `10xxxxxx` で始まるため、バイト列の途中から読み始めても  
どこが文字の先頭かを判別できます（自己同期性）。

**例：`é`（U+00E9 → 11bit: 000 1110 1001₂）→ 2バイト**
```
2バイト形式:   110xxxxx  10xxxxxx
ビットを詰める: 110[00011]  10[101001]
             = 0xC3       = 0xA9
```

**例：`あ`（U+3042 → 16bit: 0011 0000 0100 0010₂）→ 3バイト**
```
3バイト形式:   1110xxxx  10xxxxxx  10xxxxxx
ビットを詰める: 1110[0011]  10[000001]  10[000010]
             = 0xE3       = 0x81       = 0x82
```


In [4]:
# str.encode('utf-8') → bytes（バイト列）
# bytes.hex(' ')      → '41' や 'e3 81 82' のような 16 進文字列

for ch in ['!', 'A', 'é', 'δ', 'あ', '京', '🤖']:
    b = ch.encode('utf-8')
    print(f"'{ch}'  U+{ord(ch):04X}  →  {b.hex(' '):<15s}  ({len(b)} バイト)")


'!'  U+0021  →  21               (1 バイト)
'A'  U+0041  →  41               (1 バイト)
'é'  U+00E9  →  c3 a9            (2 バイト)
'δ'  U+03B4  →  ce b4            (2 バイト)
'あ'  U+3042  →  e3 81 82         (3 バイト)
'京'  U+4EAC  →  e4 ba ac         (3 バイト)
'🤖'  U+1F916  →  f0 9f a4 96      (4 バイト)


### まとめ：文字 → コードポイント → バイト列

```
文字  ('京')
  ↓ ord()          # Unicode コードポイントを取得
U+4EAC  (= 0x4EAC = 20140)
  ↓ .encode('utf-8')  # UTF-8 でバイト列に変換
e4 ba ac  (3 バイト)
```

後ほど定義する `show_tokens()` の `hex_bytes` 列はこの **UTF-8 バイト列** を  
トークンの境界で分割して表示したものです。  
全トークンの `hex_bytes` を連結すると、元テキストの UTF-8 バイト列に一致します。

---
## 1. トークナイザーの読み込み

Qwen3-4B のトークナイザーを Hugging Face cache から読み込みます。  
正常に読み込めれば、クラス名・語彙サイズ・最大系列長が表示されます。

In [5]:
import logging
from transformers import AutoTokenizer
import pandas as pd

# HuggingFace Hub の認証警告を抑制（cache から読む場合は不要なため）
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

MODEL_ID = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("tokenizer クラス:", type(tokenizer).__name__)
print("語彙サイズ:", tokenizer.vocab_size)
print("model_max_length:", tokenizer.model_max_length)

tokenizer クラス: Qwen2Tokenizer
語彙サイズ: 151643
model_max_length: 131072


---
## 2. encode / decode とトークン分割を観察する

`encode` / `decode` の基本操作を確認してから、`show_tokens()` ヘルパーを使って
テキストがどのトークンに分割されるかを様々な例で観察します。

### encode と decode の基本

In [6]:
text = "京都大学の情報学科"

ids = tokenizer.encode(text)
print("テキスト:", text)
print("token IDs:", ids)
print("トークン数:", len(ids))

テキスト: 京都大学の情報学科
token IDs: [115806, 99562, 15767, 134481, 104391]
トークン数: 5


### decode：token ID 列 → テキスト

In [7]:
# 元に戻せるか確認
restored = tokenizer.decode(ids)
print("復元:", restored)

復元: 京都大学の情報学科


In [8]:
# 1トークンずつ decode して区切りを見る
print(f"{'pos':>4}  {'token_id':>8}  decoded")
print("-" * 30)
for pos, tid in enumerate(ids):
    decoded = tokenizer.decode([tid])
    print(f"{pos:>4}  {tid:>8}  {decoded!r}")


 pos  token_id  decoded
------------------------------
   0    115806  '京都'
   1     99562  '大学'
   2     15767  'の'
   3    134481  '情報'
   4    104391  '学科'


### show_tokens() — トークン分割の詳細表示

`show_tokens()` は以下の列を持つ DataFrame を返します。

| 列 | 内容 |
|---|---|
| `token_id` | トークン ID（整数） |
| `decoded` | そのトークン単独をデコードしたテキスト |
| `piece` | トークナイザー内部の文字列表現 |
| `hex_bytes` | 内部生バイト列（UTF-8 バイト境界をまたぐトークンの確認に使う） |




> 以下の「準備１」「準備２」は `show_tokens()` の実装です。**実行するだけで OK です。**  
> コードの詳細は理解しなくて構いません。  
> **「トークン化の詳細を実例でみていく」まで読み飛ばしても構いません。**


#### 準備１：バイト変換表の作成

In [9]:
def _build_byte_maps() -> tuple[dict[int, str], dict[str, int]]:
    """
    GPT-2 開発において考案された「バイト ↔ Unicode 文字の全単射マッピング」を構築する。

    256 バイト全体を 256 個の表示可能 Unicode 文字に 1 対 1 対応させる規則:
      0x21〜0x7E, 0xA1〜0xAC, 0xAE〜0xFF (計188個): コードポイント = バイト値（自己対応）
      残り 68 バイト (0x00〜0x20, 0x7F, 0x80〜0xA0, 0xAD) : U+0100 以降に順番に割り当て

    Returns
    -------
    byte_to_char : dict[int, str]
        バイト値（0〜255）→ piece で使われる Unicode 文字
        語彙構築の方向（バイト列 → piece 文字列）
    char_to_byte : dict[str, int]
        piece の Unicode 文字 → 元のバイト値
        piece 解読の方向（piece 文字列 → バイト列）
    """
    byte_vals = (
        list(range(0x21, 0x7F))    # 0x21〜0x7E: ASCII 印字可能
        + list(range(0xA1, 0xAD))  # 0xA1〜0xAC: Latin-1 前半
        + list(range(0xAE, 0x100)) # 0xAE〜0xFF: Latin-1 後半
    )
    codepoints = byte_vals[:]  # 自己対応: コードポイント = バイト値

    n = 0
    for b in range(256):
        if b not in byte_vals:
            byte_vals.append(b)
            codepoints.append(0x100 + n)
            n += 1

    byte_to_char = {b: chr(cp) for b, cp in zip(byte_vals, codepoints)}
    char_to_byte = {chr(cp): b  for b, cp in zip(byte_vals, codepoints)}
    return byte_to_char, char_to_byte

byte_to_char, char_to_byte = _build_byte_maps()

**順方向と逆方向の変換表を生で見てみる**

In [10]:
byte_to_char

{33: '!',
 34: '"',
 35: '#',
 36: '$',
 37: '%',
 38: '&',
 39: "'",
 40: '(',
 41: ')',
 42: '*',
 43: '+',
 44: ',',
 45: '-',
 46: '.',
 47: '/',
 48: '0',
 49: '1',
 50: '2',
 51: '3',
 52: '4',
 53: '5',
 54: '6',
 55: '7',
 56: '8',
 57: '9',
 58: ':',
 59: ';',
 60: '<',
 61: '=',
 62: '>',
 63: '?',
 64: '@',
 65: 'A',
 66: 'B',
 67: 'C',
 68: 'D',
 69: 'E',
 70: 'F',
 71: 'G',
 72: 'H',
 73: 'I',
 74: 'J',
 75: 'K',
 76: 'L',
 77: 'M',
 78: 'N',
 79: 'O',
 80: 'P',
 81: 'Q',
 82: 'R',
 83: 'S',
 84: 'T',
 85: 'U',
 86: 'V',
 87: 'W',
 88: 'X',
 89: 'Y',
 90: 'Z',
 91: '[',
 92: '\\',
 93: ']',
 94: '^',
 95: '_',
 96: '`',
 97: 'a',
 98: 'b',
 99: 'c',
 100: 'd',
 101: 'e',
 102: 'f',
 103: 'g',
 104: 'h',
 105: 'i',
 106: 'j',
 107: 'k',
 108: 'l',
 109: 'm',
 110: 'n',
 111: 'o',
 112: 'p',
 113: 'q',
 114: 'r',
 115: 's',
 116: 't',
 117: 'u',
 118: 'v',
 119: 'w',
 120: 'x',
 121: 'y',
 122: 'z',
 123: '{',
 124: '|',
 125: '}',
 126: '~',
 161: '¡',
 162: '¢',
 163: '£',

In [11]:
char_to_byte

{'!': 33,
 '"': 34,
 '#': 35,
 '$': 36,
 '%': 37,
 '&': 38,
 "'": 39,
 '(': 40,
 ')': 41,
 '*': 42,
 '+': 43,
 ',': 44,
 '-': 45,
 '.': 46,
 '/': 47,
 '0': 48,
 '1': 49,
 '2': 50,
 '3': 51,
 '4': 52,
 '5': 53,
 '6': 54,
 '7': 55,
 '8': 56,
 '9': 57,
 ':': 58,
 ';': 59,
 '<': 60,
 '=': 61,
 '>': 62,
 '?': 63,
 '@': 64,
 'A': 65,
 'B': 66,
 'C': 67,
 'D': 68,
 'E': 69,
 'F': 70,
 'G': 71,
 'H': 72,
 'I': 73,
 'J': 74,
 'K': 75,
 'L': 76,
 'M': 77,
 'N': 78,
 'O': 79,
 'P': 80,
 'Q': 81,
 'R': 82,
 'S': 83,
 'T': 84,
 'U': 85,
 'V': 86,
 'W': 87,
 'X': 88,
 'Y': 89,
 'Z': 90,
 '[': 91,
 '\\': 92,
 ']': 93,
 '^': 94,
 '_': 95,
 '`': 96,
 'a': 97,
 'b': 98,
 'c': 99,
 'd': 100,
 'e': 101,
 'f': 102,
 'g': 103,
 'h': 104,
 'i': 105,
 'j': 106,
 'k': 107,
 'l': 108,
 'm': 109,
 'n': 110,
 'o': 111,
 'p': 112,
 'q': 113,
 'r': 114,
 's': 115,
 't': 116,
 'u': 117,
 'v': 118,
 'w': 119,
 'x': 120,
 'y': 121,
 'z': 122,
 '{': 123,
 '|': 124,
 '}': 125,
 '~': 126,
 '¡': 161,
 '¢': 162,
 '£': 163,

In [12]:
# ── マッピングの詳細確認 ────────────────────────────────────────────────────

# 順方向（byte_to_char）: バイト値 → Unicode 文字
print("▼ 順方向（byte_to_char）: バイト値 → piece に現れる Unicode 文字")
print("  ASCII 印字可能（自己対応）:")
for bval in [0x21, 0x41, 0x7E]:
    ch = byte_to_char[bval]
    print(f"    byte_to_char[0x{bval:02X}] = {ch!r}  (U+{ord(ch):04X})")
print("  直接対応できない 68 バイト（U+0100 以降に割り当て）:")
for bval, label in [(0x00, "NUL"), (0x0A, "LF"), (0x20, "SP"),
                    (0x7F, "DEL"), (0x80, ""), (0xA0, "NBSP"), (0xAD, "SHY")]:
    ch = byte_to_char[bval]
    print(f"    byte_to_char[0x{bval:02X}] ({label:4s}) = {ch!r}  (U+{ord(ch):04X})")
print()

# 逆方向（char_to_byte）: Unicode 文字 → バイト値
print("▼ 逆方向（char_to_byte）: piece の Unicode 文字 → 元のバイト値")
for ch in ["!", "A", "ã", "ĥ", "Ī"]:
    bval = char_to_byte[ch]
    print(f"    char_to_byte[{ch!r}]  (U+{ord(ch):04X})  →  byte 0x{bval:02X}")
print()

# 具体例：piece "ãĥĪ"（「ト」を表す piece）をバイト列に戻す
raw_example = "ãĥĪ"
print(f"▼ 具体例: piece {raw_example!r} をバイト列に戻す  （char_to_byte を使う）")
result = bytes([char_to_byte[c] for c in raw_example])
for ch, bval in zip(raw_example, result):
    print(f"    char_to_byte[{ch!r}]  (U+{ord(ch):04X})  →  byte 0x{bval:02X}")
print(f"  → {result.hex(' ')}  →  UTF-8 decode: {result.decode('utf-8')!r}")


▼ 順方向（byte_to_char）: バイト値 → piece に現れる Unicode 文字
  ASCII 印字可能（自己対応）:
    byte_to_char[0x21] = '!'  (U+0021)
    byte_to_char[0x41] = 'A'  (U+0041)
    byte_to_char[0x7E] = '~'  (U+007E)
  直接対応できない 68 バイト（U+0100 以降に割り当て）:
    byte_to_char[0x00] (NUL ) = 'Ā'  (U+0100)
    byte_to_char[0x0A] (LF  ) = 'Ċ'  (U+010A)
    byte_to_char[0x20] (SP  ) = 'Ġ'  (U+0120)
    byte_to_char[0x7F] (DEL ) = 'ġ'  (U+0121)
    byte_to_char[0x80] (    ) = 'Ģ'  (U+0122)
    byte_to_char[0xA0] (NBSP) = 'ł'  (U+0142)
    byte_to_char[0xAD] (SHY ) = 'Ń'  (U+0143)

▼ 逆方向（char_to_byte）: piece の Unicode 文字 → 元のバイト値
    char_to_byte['!']  (U+0021)  →  byte 0x21
    char_to_byte['A']  (U+0041)  →  byte 0x41
    char_to_byte['ã']  (U+00E3)  →  byte 0xE3
    char_to_byte['ĥ']  (U+0125)  →  byte 0x83
    char_to_byte['Ī']  (U+012A)  →  byte 0x88

▼ 具体例: piece 'ãĥĪ' をバイト列に戻す  （char_to_byte を使う）
    char_to_byte['ã']  (U+00E3)  →  byte 0xE3
    char_to_byte['ĥ']  (U+0125)  →  byte 0x83
    char_to_byte['Ī']  (U+012A)  → 

#### 準備２：トークナイザー説明用コード

In [13]:
# ── 生バイト取得ヘルパー ────────────────────────────────────────────────────
#
# char_to_byte を使って piece 文字列を元のバイト列に戻す。
# piece.encode("utf-8") ではなく piece から直接変換することで、
# decode/re-encode による余計な変換を避ける。
#
def _piece_to_bytes(piece: str) -> bytes:
    return bytes([char_to_byte[c] for c in piece])


# ── show_tokens 本体 ────────────────────────────────────────────────────────
def show_tokens(text: str):

    # テキストをトークン ID 列に変換
    ids = tokenizer.encode(text)

    # decoded: token ID を単独でデコードしたテキストの列
    # UTF-8 文字境界をまたぐトークンでは不完全なバイト列が U+FFFD（□）になる。
    decoded = [tokenizer.decode([tid]) for tid in ids]

    # pieces: トークナイザー内部の文字列表現（Ġ = スペースなど）の列
    pieces = tokenizer.convert_ids_to_tokens(ids)

    # hex_bytes: piece から char_to_byte で直接変換した内部生バイト列。
    # 全トークンの hex_bytes を連結すると元テキストの UTF-8 バイト列に一致する。
    hex_bytes = [_piece_to_bytes(piece).hex(" ") for piece in pieces]

    print("utf-8 バイト列: " + text.encode("utf-8").hex(" "))

    return pd.DataFrame({
        "pos": range(len(ids)),
        "token_id": ids,
        "decoded": [repr(d) for d in decoded],
        "piece": pieces,
        "hex_bytes": hex_bytes,
    }).set_index("pos")


#### トークン化の詳細を実例でみていく

**日本語：漢字・かな**

In [14]:
show_tokens("京都大学の情報学科")

utf-8 バイト列: e4 ba ac e9 83 bd e5 a4 a7 e5 ad a6 e3 81 ae e6 83 85 e5 a0 b1 e5 ad a6 e7 a7 91


,token_id,decoded,piece,hex_bytes
pos,,,,
0,115806,'京都',äº¬éĥ½,e4 ba ac e9 83 bd
1,99562,'大学',å¤§åŃ¦,e5 a4 a7 e5 ad a6
2,15767,'の',ãģ®,e3 81 ae
3,134481,'情報',æĥħåł±,e6 83 85 e5 a0 b1
4,104391,'学科',åŃ¦ç§ĳ,e5 ad a6 e7 a7 91


In [15]:
show_tokens("東京")

utf-8 バイト列: e6 9d b1 e4 ba ac


,token_id,decoded,piece,hex_bytes
pos,,,,
0,102356,'東',æĿ±,e6 9d b1
1,46553,'京',äº¬,e4 ba ac


In [16]:
show_tokens("京都")

utf-8 バイト列: e4 ba ac e9 83 bd


,token_id,decoded,piece,hex_bytes
pos,,,,
0,115806,'京都',äº¬éĥ½,e4 ba ac e9 83 bd


In [17]:
show_tokens("言語モデルとは何か")

utf-8 バイト列: e8 a8 80 e8 aa 9e e3 83 a2 e3 83 87 e3 83 ab e3 81 a8 e3 81 af e4 bd 95 e3 81 8b


,token_id,decoded,piece,hex_bytes
pos,,,,
0,77144,'言',è¨Ģ,e8 a8 80
1,102819,'語',èªŀ,e8 aa 9e
2,134680,'モデル',ãĥ¢ãĥĩãĥ«,e3 83 a2 e3 83 87 e3 83 ab
3,126238,'とは',ãģ¨ãģ¯,e3 81 a8 e3 81 af
4,130967,'何か',ä½ķãģĭ,e4 bd 95 e3 81 8b


In [18]:
show_tokens("情報AI基礎")

utf-8 バイト列: e6 83 85 e5 a0 b1 41 49 e5 9f ba e7 a4 8e


,token_id,decoded,piece,hex_bytes
pos,,,,
0,134481,'情報',æĥħåł±,e6 83 85 e5 a0 b1
1,15469,'AI',AI,41 49
2,112952,'基礎',åŁºç¤İ,e5 9f ba e7 a4 8e


**英語とカタカナ**

同じ概念でも表記が違うとトークン数が大きく変わります。

In [19]:
show_tokens("transformer")

utf-8 バイト列: 74 72 61 6e 73 66 6f 72 6d 65 72


,token_id,decoded,piece,hex_bytes
pos,,,,
0,4701,'transform',transform,74 72 61 6e 73 66 6f 72 6d
1,261,'er',er,65 72


In [20]:
# トークン境界とutf-8境界が一致しない例
show_tokens("トランスフォーマー")

utf-8 バイト列: e3 83 88 e3 83 a9 e3 83 b3 e3 82 b9 e3 83 95 e3 82 a9 e3 83 bc e3 83 9e e3 83 bc


,token_id,decoded,piece,hex_bytes
pos,,,,
0,19674,'ト',ãĥĪ,e3 83 88
1,128062,'ランス',ãĥ©ãĥ³ãĤ¹,e3 83 a9 e3 83 b3 e3 82 b9
2,125964,'フォ',ãĥķãĤ©,e3 83 95 e3 82 a9
3,24041,'ー�',ãĥ¼ãĥ,e3 83 bc e3 83
4,252,'�',ŀ,9e
5,11710,'ー',ãĥ¼,e3 83 bc


**英語：先頭スペースあり・なしで別トークンになる**

英文中の単語は通常スペースに続くため、スペースをトークンに吸収した形式が使われる。  
Section 0 の `byte_to_char` で `byte 0x20 (SP) → Ġ` と対応しており、
`Ġcapital` と `capital` は異なるトークン ID を持つ。

- `Ġword`（スペースあり）: 文中に現れる単語の通常形
- `word`（スペースなし）: 文頭や別の文字に続く場合

In [21]:
show_tokens(" capital")

utf-8 バイト列: 20 63 61 70 69 74 61 6c


,token_id,decoded,piece,hex_bytes
pos,,,,
0,6722,' capital',Ġcapital,20 63 61 70 69 74 61 6c


In [22]:
show_tokens("capital")

utf-8 バイト列: 63 61 70 69 74 61 6c


,token_id,decoded,piece,hex_bytes
pos,,,,
0,65063,'capital',capital,63 61 70 69 74 61 6c


In [23]:
show_tokens(" Tokyo")

utf-8 バイト列: 20 54 6f 6b 79 6f


,token_id,decoded,piece,hex_bytes
pos,,,,
0,26194,' Tokyo',ĠTokyo,20 54 6f 6b 79 6f


In [24]:
show_tokens("Tokyo")

utf-8 バイト列: 54 6f 6b 79 6f


,token_id,decoded,piece,hex_bytes
pos,,,,
0,52854,'Tok',Tok,54 6f 6b
1,16032,'yo',yo,79 6f


**英語：大文字，小文字の違いは別トークンになる**

In [25]:
show_tokens("TOKYO")

utf-8 バイト列: 54 4f 4b 59 4f


,token_id,decoded,piece,hex_bytes
pos,,,,
0,51,'T',T,54
1,3925,'OK',OK,4f 4b
2,56644,'YO',YO,59 4f


In [26]:
show_tokens("tokyo")

utf-8 バイト列: 74 6f 6b 79 6f


,token_id,decoded,piece,hex_bytes
pos,,,,
0,29594,'tok',tok,74 6f 6b
1,16032,'yo',yo,79 6f


**記号・絵文字・数字**

絵文字は 4 バイトの UTF-8 だが 1 トークンに収まることが多い。
数字は 1 桁ずつトークンになりやすい。

In [27]:
show_tokens("🤖🎌")

utf-8 バイト列: f0 9f a4 96 f0 9f 8e 8c


,token_id,decoded,piece,hex_bytes
pos,,,,
0,147238,'🤖',ðŁ¤ĸ,f0 9f a4 96
1,148192,'🎌',ðŁİĮ,f0 9f 8e 8c


In [28]:
show_tokens("1234567890")

utf-8 バイト列: 31 32 33 34 35 36 37 38 39 30


,token_id,decoded,piece,hex_bytes
pos,,,,
0,16,'1',1,31
1,17,'2',2,32
2,18,'3',3,33
3,19,'4',4,34
4,20,'5',5,35
5,21,'6',6,36
6,22,'7',7,37
7,23,'8',8,38
8,24,'9',9,39


---
## 3. special token を確認する

Qwen3 の chat template では `<|im_start|>` や `<|im_end|>` などの special token が使われます。  
これらは通常のテキストとは別に語彙に登録されています。

In [29]:
print("special tokens:")
for name, tok in tokenizer.special_tokens_map.items():
    print(f"  {name}: {tok}  id={tokenizer.convert_tokens_to_ids(tok)}")

special tokens:
  eos_token: <|im_end|>  id=151645
  pad_token: <|endoftext|>  id=151643


In [30]:
# added_tokens_encoder には special token を含む全追加トークンが入っている
print("追加トークン数:", len(tokenizer.added_tokens_encoder),"\n")
items = sorted(tokenizer.added_tokens_encoder.items(), key=lambda x: x[1])
for tok, tid in items:
    print(f"  id={tid:>7}  {tok}")

追加トークン数: 26 

  id= 151643  <|endoftext|>
  id= 151644  <|im_start|>
  id= 151645  <|im_end|>
  id= 151646  <|object_ref_start|>
  id= 151647  <|object_ref_end|>
  id= 151648  <|box_start|>
  id= 151649  <|box_end|>
  id= 151650  <|quad_start|>
  id= 151651  <|quad_end|>
  id= 151652  <|vision_start|>
  id= 151653  <|vision_end|>
  id= 151654  <|vision_pad|>
  id= 151655  <|image_pad|>
  id= 151656  <|video_pad|>
  id= 151657  <tool_call>
  id= 151658  </tool_call>
  id= 151659  <|fim_prefix|>
  id= 151660  <|fim_middle|>
  id= 151661  <|fim_suffix|>
  id= 151662  <|fim_pad|>
  id= 151663  <|repo_name|>
  id= 151664  <|file_sep|>
  id= 151665  <tool_response>
  id= 151666  </tool_response>
  id= 151667  <think>
  id= 151668  </think>


Qwen3-4B の追加トークンは用途別に分類できます。

| カテゴリ | トークン例 | 用途 |
|---|---|---|
| チャット制御 | `<\|im_start\|>` `<\|im_end\|>` `<\|endoftext\|>` | ロール境界・系列終端 |
| 物体参照・座標 | `<\|object_ref_*\|>` `<\|box_*\|>` `<\|quad_*\|>` | 画像内の位置指定（Grounding） |
| マルチモーダル | `<\|vision_*\|>` `<\|image_pad\|>` `<\|video_pad\|>` | 画像・動画トークンの埋め込み |
| ツール・その他 | `<\|tool_call\|>` など | 関数呼び出しなど |

テキストのみの用途では、チャット制御トークンだけを意識すれば十分です。

---
## 4. chat template の適用とトークン化

chat template を適用しトークン化して，トークン単位の表として可視化します。

### テンプレートの適用

In [31]:
messages = [
    {"role": "user", "content": "言語モデルとは何か、一言で教えてください。"}
]

chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

# non-thinking　モードの例
print(chat_text)

<|im_start|>user
言語モデルとは何か、一言で教えてください。<|im_end|>
<|im_start|>assistant
<think>

</think>




`enable_thinking=False` でも生成プロンプトの末尾に `<think>\n\n</think>\n\n` が含まれます。  
これは Qwen3 の **non-thinking モードの仕様** です。  
空の `<think>` ブロックをあらかじめ埋めておくことで、モデルが thinking をスキップして直接応答に移るよう誘導しています。

`enable_thinking=True` にすると、生成プロンプトは `<|im_start|>assistant\n` だけで終わります。  
モデルが自分で `<think>` ブロックを生成してから応答に移ります。

In [32]:
# enable_thinking=True との比較
chat_text_thinking = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

# thinking モードの例
print(chat_text_thinking)

<|im_start|>user
言語モデルとは何か、一言で教えてください。<|im_end|>
<|im_start|>assistant



### トークン化

In [33]:
show_tokens(chat_text)

utf-8 バイト列: 3c 7c 69 6d 5f 73 74 61 72 74 7c 3e 75 73 65 72 0a e8 a8 80 e8 aa 9e e3 83 a2 e3 83 87 e3 83 ab e3 81 a8 e3 81 af e4 bd 95 e3 81 8b e3 80 81 e4 b8 80 e8 a8 80 e3 81 a7 e6 95 99 e3 81 88 e3 81 a6 e3 81 8f e3 81 a0 e3 81 95 e3 81 84 e3 80 82 3c 7c 69 6d 5f 65 6e 64 7c 3e 0a 3c 7c 69 6d 5f 73 74 61 72 74 7c 3e 61 73 73 69 73 74 61 6e 74 0a 3c 74 68 69 6e 6b 3e 0a 0a 3c 2f 74 68 69 6e 6b 3e 0a 0a


,token_id,decoded,piece,hex_bytes
pos,,,,
0,151644,'<|im_start|>',<|im_start|>,3c 7c 69 6d 5f 73 74 61 72 74 7c 3e
1,872,'user',user,75 73 65 72
2,198,'\n',Ċ,0a
3,77144,'言',è¨Ģ,e8 a8 80
4,102819,'語',èªŀ,e8 aa 9e
5,134680,'モデル',ãĥ¢ãĥĩãĥ«,e3 83 a2 e3 83 87 e3 83 ab
6,126238,'とは',ãģ¨ãģ¯,e3 81 a8 e3 81 af
7,130967,'何か',ä½ķãģĭ,e4 bd 95 e3 81 8b
8,5373,'、',ãĢģ,e3 80 81


---
## 5. 語彙の先頭と末尾を覗く

トークン ID の順序には意味があります。  
先頭付近（ID 0〜）には基本的な文字・記号が、末尾付近（`vocab_size` 直前）には通常語彙の最後が並びます。  
`vocab_size` 以降が `added_tokens_encoder` に登録された特殊トークンです。

In [34]:
# 語彙の先頭（ID 0〜19）
rows = []
for tid in range(20):
    piece   = tokenizer.convert_ids_to_tokens([tid])[0]
    decoded = tokenizer.decode([tid])
    rows.append({"token_id": tid, "decoded": repr(decoded), "piece": piece})
pd.DataFrame(rows).set_index("token_id")


,decoded,piece
token_id,,
0,'!',!
1,"'""'",""""
2,'#',#
3,'$',$
4,'%',%
5,'&',&
6,"""'""",'
7,'(',(
8,')',)


In [35]:
# 通常語彙末尾と added token の境界付近
boundary = tokenizer.vocab_size
print(f"vocab_size = {boundary}  （ID {boundary} 以降が added token）")

rows = []
for tid in range(boundary - 3, boundary + 4):
    piece   = tokenizer.convert_ids_to_tokens([tid])[0]
    decoded = tokenizer.decode([tid])
    rows.append({"token_id": tid, "decoded": repr(decoded), "piece": piece})
pd.DataFrame(rows).set_index("token_id")


vocab_size = 151643  （ID 151643 以降が added token）


,decoded,piece
token_id,,
151640,'⍨',âį¨
151641,'⺟',âºŁ
151642,'⽗',â½Ĺ
151643,'<|endoftext|>',<|endoftext|>
151644,'<|im_start|>',<|im_start|>
151645,'<|im_end|>',<|im_end|>
151646,'<|object_ref_start|>',<|object_ref_start|>


---
## まとめ

| 操作 | 関数 | 入力 → 出力 |
|---|---|---|
| テキスト → ID列 | `tokenizer.encode()` | `str` → `list[int]` |
| ID列 → テキスト | `tokenizer.decode()` | `list[int]` → `str` |
| ID列 → piece列 | `tokenizer.convert_ids_to_tokens()` | `list[int]` → `list[str]` |
| chat template 適用 | `tokenizer.apply_chat_template()` | `list[dict]` → `str` |

**観察のポイント：**
- 日本語は1文字 = 1トークンとは限らない（複数文字がまとめられることも）
- 英語は単語単位が多いが、接尾辞・接頭辞で分割されることもある
- 数字は1桁ずつ分割されることが多い
- 絵文字はバイト列に分解されることがある
- special token は通常の語彙とは別管理

次のノートブックでは **forward pass**（モデルが入力を処理する一連の計算）を観察します。